# NCA -- Growing Neural Cellular Automata

Mordvintsev, Randazzo, Niklasson, Levin, *Growing Neural Cellular Automata: Differentiable Model of Morphogenesis*, Distill 2020 ([distill.pub/2020/growing-ca](https://distill.pub/2020/growing-ca/)).

Every other model in this repo does one forward pass: image in, label out. NCA is different: the learned object is a *local update rule* applied identically at every cell of a grid, for many stochastic asynchronous steps. A single alive seed cell, iterated under this rule, self-organizes into a target pattern. See `model.py` for `perceive` (fixed Sobel/identity kernels), the learned update net, stochastic firing, and alive-masking.

This repo trains against a procedurally generated RGBA target (a simple radial "flower") instead of the paper's emoji, to avoid an external asset/licensing dependency -- see the honesty note in `model.py`.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import NCAModel, make_target, seed_state

set_seed(0)
device = resolve_device('auto')
print('device:', device)

In [ ]:
grid_size = 40
batch_size = 8
target = make_target(grid_size).to(device)
target_batch = target.unsqueeze(0).expand(batch_size, -1, -1, -1)

plt.imshow((target[:3] * target[3:4] + torch.ones(3,1,1) * (1 - target[3:4])).permute(1,2,0))
plt.title('training target')
plt.axis('off')
plt.show()

In [ ]:
model = NCAModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

iterations = 300  # a reduced schedule for a quick notebook run; the paper trains ~8000 iterations
history = []
for it in range(1, iterations + 1):
    state = seed_state(batch_size, grid_size, device=device)
    n_steps = torch.randint(48, 65, (1,)).item()
    opt.zero_grad()
    final_state = model(state, n_steps)
    loss = F.mse_loss(final_state[:, :4], target_batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    history.append(loss.item())

print(f'final loss: {history[-1]:.4f}')
plt.plot(history); plt.xlabel('iteration'); plt.ylabel('MSE to target'); plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    state = seed_state(1, grid_size, device=device)
    checkpoints = [0, 8, 20, 36, 56]
    snapshots, step = [], 0
    for cp in checkpoints:
        while step < cp:
            state = model.step(state)
            step += 1
        rgb = state[0, :3].clamp(0, 1).cpu()
        alpha = state[0, 3:4].clamp(0, 1).cpu()
        snapshots.append((rgb * alpha + torch.ones(3,1,1) * (1 - alpha)).permute(1,2,0).numpy())

fig, axes = plt.subplots(1, len(checkpoints), figsize=(2.2*len(checkpoints), 2.4))
for ax, snap, cp in zip(axes, snapshots, checkpoints):
    ax.imshow(snap); ax.set_title(f'step {cp}'); ax.axis('off')
fig.tight_layout()
plt.show()